## Tensor基础

In [1]:
import torch

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("形状:", x.shape)
print("数据类型:", x.dtype)
print("内容:\n", x)

形状: torch.Size([2, 2])
数据类型: torch.float32
内容:
 tensor([[1., 2.],
        [3., 4.]])


### 第二块--搬到GPU

In [8]:
x_gpu = x.cuda()
print("设备：", x_gpu.device)

设备： cuda:0


### 第三块--常用操作

In [7]:
a = torch.ones(3, 3)
b = torch.zeros(3, 3)
print("相加：\n", a + b)
print("矩阵乘法：\n", a @ b)
print("改变形状：", a.reshape(1,9).shape)

相加：
 tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
矩阵乘法：
 tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])
改变形状： torch.Size([1, 9])


## 第三步--自动求导（核心）

In [11]:
# x 是一个需要计算梯度的变量
x = torch.tensor(3.0, requires_grad=True)

# 做一个计算 y = x平方
y = x ** 2

# 反向传播，计算 dy/dx
y.backward()

# 查看梯度 dy/dx = 2x = 2*3 = 6
print("x的梯度：", x.grad)

x的梯度： tensor(6.)


## 第四步--手写训练循环

In [13]:
import torch
import torch.nn as nn

# --1. 造假数据 ----------------------
# 100个像本，每个样本两个特征
X = torch.randn(100, 2)
# 标签：两个特征之和大于0就是1 ，否则是0
y = (X[:, 0] + X[:,1] > 0).float()

print("数据形状", X.shape)
print("标签形状", y.shape)
print("正样本数量", y.sum().item())

数据形状 torch.Size([100, 2])
标签形状 torch.Size([100])
正样本数量 50.0


In [15]:
# --2. 定义模型---------------
model = nn.Sequential(
    nn.Linear(2,8),
    nn.ReLU(),
    nn.Linear(8,1),
    nn.Sigmoid()

)

In [16]:
# --3. 损失函数 + 优化器
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

In [17]:
# -- 4. 训练循环 ------------------------
for epoch in range(100):
    #向前传播：数据流过模型得到预测值
    pred = model(X)

    #计算loss：预测值和真实值的差距
    loss = criterion(pred.squeeze(), y)

    #清空上一步的梯度（必须做否则梯度会增加）
    optimizer.zero_grad()

    # 反向传播：计算每个参数的梯度
    loss.backward()

    # 更新参数：沿梯度方向调整参数
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}, Loss:{loss.item():.4f}")

Epoch   0, Loss:0.6249
Epoch  10, Loss:0.4842
Epoch  20, Loss:0.3612
Epoch  30, Loss:0.2638
Epoch  40, Loss:0.1948
Epoch  50, Loss:0.1493
Epoch  60, Loss:0.1208
Epoch  70, Loss:0.1019
Epoch  80, Loss:0.0885
Epoch  90, Loss:0.0783
